# SochDB vs ChromaDB — Practical Showcase

**Run every cell top-to-bottom in a fresh kernel.**

| Test | What it measures | Why it's fair |
|------|-----------------|---------------|
| 1 — Insert speed | Persistent DB insert throughput | Both use persistent on-disk storage |
| 2 — Query latency | Per-query search time | Same persistent index, same k=10 |
| 3 — Search quality | Top-1 accuracy on 30 queries | BM25+vector vs pure-vector, same docs |
| 4 — Token efficiency | LLM context tokens for N docs | TOON vs JSON, same 25 documents |

## Cell 1 — Imports & version pins

In [1]:
import time, os, shutil, gc, json, sys, subprocess
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import tiktoken
import chromadb
from sentence_transformers import SentenceTransformer
import sochdb as _s

# ── Version record (reproducibility) ──────────────────────────────────────
import chromadb, sentence_transformers
print(f'python        : {sys.version.split()[0]}')
print(f'chromadb      : {chromadb.__version__}')
print(f'sochdb        : {getattr(_s, "__version__", "unknown")}')
print(f'sentence-transformers: {sentence_transformers.__version__}')
print(f'numpy         : {np.__version__}')

def _get(name): return getattr(_s, name, None)
Database         = _get('Database')
VectorIndex      = _get('VectorIndex')
CollectionConfig = _get('CollectionConfig')
DistanceMetric   = _get('DistanceMetric')

DIM   = 384
SEED  = 42
enc   = tiktoken.get_encoding('cl100k_base')
rng   = np.random.default_rng(SEED)   # single RNG used everywhere

# Fixed embedding model — pin by revision for reproducibility
MODEL_NAME = 'sentence-transformers/all-MiniLM-L6-v2'
print(f'\nLoading {MODEL_NAME}...')
model = SentenceTransformer(MODEL_NAME)
print('Ready.')

# ── Temp directories ───────────────────────────────────────────────────────
CHROMA_BENCH_DIR  = './tmp_chroma_bench'
CHROMA_SEARCH_DIR = './tmp_chroma_search'
SOCHDB_BENCH_DIR  = './tmp_sochdb_bench'
SOCHDB_SEARCH_DIR = './tmp_sochdb_search'

def wipe(*dirs):
    """Delete dirs and wait for OS to release any lingering handles."""
    for d in dirs:
        if os.path.exists(d):
            shutil.rmtree(d)
    gc.collect()
    time.sleep(0.3)

# Start clean
wipe(CHROMA_BENCH_DIR, CHROMA_SEARCH_DIR, SOCHDB_BENCH_DIR, SOCHDB_SEARCH_DIR)
print('Temp dirs cleared.')


python        : 3.14.3
chromadb      : 1.5.4
sochdb        : 0.5.4
sentence-transformers: 5.2.3
numpy         : 2.4.2

Loading sentence-transformers/all-MiniLM-L6-v2...


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Ready.
Temp dirs cleared.


---
## Cell 2 — Dataset (25 documents, 6 topics)

In [2]:
DOCUMENTS = [
    {'id':'doc_001','text':'Attention Is All You Need introduced the Transformer architecture replacing RNNs with self-attention for sequence modeling.','topic':'transformers'},
    {'id':'doc_002','text':'BERT pre-trains on masked language modeling and next-sentence prediction using bidirectional encoder representations.','topic':'transformers'},
    {'id':'doc_003','text':'GPT-4o is OpenAI multimodal model handling text image and audio with a single unified architecture and end-to-end training.','topic':'llm'},
    {'id':'doc_004','text':'Flash Attention reduces memory complexity from O n squared to O n by tiling and recomputation on GPU SRAM.','topic':'optimization'},
    {'id':'doc_005','text':'Rotary Position Embeddings RoPE encode positional information into query and key matrices without additive positional tokens.','topic':'transformers'},
    {'id':'doc_006','text':'LoRA Low-Rank Adaptation freezes pretrained weights and injects trainable rank-decomposition matrices for efficient finetuning.','topic':'finetuning'},
    {'id':'doc_007','text':'RLHF Reinforcement Learning from Human Feedback trains a reward model on human comparisons then uses PPO to align LLM outputs.','topic':'alignment'},
    {'id':'doc_008','text':'Constitutional AI trains Claude to follow a set of principles using self-critique and revision before RLHF fine-tuning.','topic':'alignment'},
    {'id':'doc_009','text':'QLoRA enables finetuning of 65B parameter models on a single GPU using 4-bit NormalFloat quantization and double quantization.','topic':'finetuning'},
    {'id':'doc_010','text':'DPO Direct Preference Optimization eliminates the reward model and optimizes LLM preferences directly with a classification loss.','topic':'alignment'},
    {'id':'doc_011','text':'FAISS provides efficient GPU and CPU indexes for billion-scale nearest neighbor retrieval with product quantization.','topic':'vector-db'},
    {'id':'doc_012','text':'ChromaDB is an open-source vector database with a Python-first API supporting metadata filtering and embedding functions.','topic':'vector-db'},
    {'id':'doc_013','text':'Reciprocal Rank Fusion RRF merges ranked lists from multiple retrievers by summing reciprocal ranks for hybrid search.','topic':'retrieval'},
    {'id':'doc_014','text':'ColBERT late interaction model scores query-document pairs with token-level MaxSim between query and passage embeddings.','topic':'retrieval'},
    {'id':'doc_015','text':'RAG Retrieval Augmented Generation grounds LLM outputs by retrieving relevant documents at inference time from a vector store.','topic':'retrieval'},
    {'id':'doc_016','text':'Stable Diffusion is a latent diffusion model that generates images by iteratively denoising in a compressed latent space.','topic':'generative'},
    {'id':'doc_017','text':'ControlNet adds conditional control to Stable Diffusion by attaching trainable copies of encoder blocks to the frozen UNet.','topic':'generative'},
    {'id':'doc_018','text':'Textual Inversion learns a new text embedding token to represent a concept from just 3 to 5 reference images for diffusion.','topic':'generative'},
    {'id':'doc_019','text':'Wengrowski and Dana CVPR 2019 propose Light Field Messaging deep photographic steganography using differentiable rendering.','topic':'steganography'},
    {'id':'doc_020','text':'HiDDeN is an end-to-end deep steganography system with an adversarial noise layer simulating JPEG compression attacks.','topic':'steganography'},
    {'id':'doc_021','text':'Glaze protects artist style by adding imperceptible perturbations that shift feature representations in CLIP embedding space.','topic':'content-protection'},
    {'id':'doc_022','text':'Nightshade is a data poisoning attack that corrupts model fine-tuning by injecting semantically shifted image-text pairs.','topic':'content-protection'},
    {'id':'doc_023','text':'FGSM Fast Gradient Sign Method generates adversarial examples in a single gradient step along the sign direction.','topic':'adversarial'},
    {'id':'doc_024','text':'UAP Universal Adversarial Perturbation is an image-agnostic delta that fools classifiers across the entire dataset.','topic':'adversarial'},
    {'id':'doc_025','text':'Mixture of Experts MoE routes each token to a subset of expert FFN layers enabling scaling without proportional compute cost.','topic':'architecture'},
]

texts      = [d['text'] for d in DOCUMENTS]
doc_by_id  = {d['id']: d for d in DOCUMENTS}
embeddings = model.encode(texts, normalize_embeddings=True).astype(np.float32)
print(f'{len(DOCUMENTS)} docs embedded — shape {embeddings.shape}')

# ── Verify expected IDs exist (catch dataset drift early) ─────────────────
EXPECTED_IDS = ['doc_003','doc_007','doc_009','doc_013','doc_019','doc_024']
missing = [e for e in EXPECTED_IDS if e not in doc_by_id]
assert not missing, f'Missing expected doc IDs: {missing}'
print('All expected document IDs verified ✓')


25 docs embedded — shape (25, 384)
All expected document IDs verified ✓


---
## Tests 1 & 2 — Insert Speed + Query Latency (Persistent Storage)

**Why this is a fair comparison:** both databases write to disk. ChromaDB uses SQLite + HNSW. SochDB uses its own WAL-backed store + HNSW. Neither has an unfair in-memory advantage.

In [3]:
import sochdb, pathlib, re
ns_text = (pathlib.Path(sochdb.__file__).parent / 'namespace.py').read_text()
for m in re.finditer(r'def search\b.*\n(?:.*\n){0,15}', ns_text):
    print(m.group()); break

def search(self, request: SearchRequest) -> SearchResults:
        """
        Unified search API.
        
        This is the primary search method supporting vector, keyword,
        and hybrid search modes. Use convenience methods for simpler cases.
        
        Args:
            request: SearchRequest with query parameters
            
        Returns:
            SearchResults with matching documents
        """
        request.validate(self._config.dimension)
        
        # Determine search mode



In [6]:
def do_search(col, vec):
    return col.search(vec)

In [7]:
import inspect

N          = 5_000
BATCH_SIZE = 500
N_QUERIES  = 300

bench_ids  = [f'v{i}' for i in range(N)]
bench_vecs = rng.standard_normal((N, DIM)).astype(np.float32)
bench_vecs /= np.linalg.norm(bench_vecs, axis=1, keepdims=True)
query_vecs = rng.standard_normal((N_QUERIES, DIM)).astype(np.float32)
query_vecs /= np.linalg.norm(query_vecs, axis=1, keepdims=True)

# ══ ChromaDB ══════════════════════════════════════════════════════════════
wipe(CHROMA_BENCH_DIR)
_cc = chromadb.PersistentClient(path=CHROMA_BENCH_DIR)
col_c = _cc.get_or_create_collection('bench', metadata={'hnsw:space': 'cosine'})

print(f'Inserting {N:,} x {DIM}D vectors (batch={BATCH_SIZE})...')
t0 = time.perf_counter()
for i in range(0, N, BATCH_SIZE):
    s = slice(i, i + BATCH_SIZE)
    col_c.add(ids=bench_ids[s], embeddings=bench_vecs[s].tolist())
chroma_insert = time.perf_counter() - t0
print(f'ChromaDB insert : {chroma_insert:.3f}s')

t0 = time.perf_counter()
for qv in query_vecs:
    col_c.query(query_embeddings=[qv.tolist()], n_results=10)
chroma_query_ms = (time.perf_counter() - t0) / N_QUERIES * 1000
print(f'ChromaDB query  : {chroma_query_ms:.2f} ms/query (n={N_QUERIES})')

del col_c, _cc; gc.collect(); time.sleep(0.3)

# ══ SochDB ════════════════════════════════════════════════════════════════
wipe(SOCHDB_BENCH_DIR)
soch_db_bench = Database.open(SOCHDB_BENCH_DIR)
soch_db_bench.create_namespace('bench')

with soch_db_bench.use_namespace('bench') as ns:
    soch_col_bench = ns.create_collection(
        CollectionConfig(name='vecs', dimension=DIM, metric=DistanceMetric.COSINE)
    )

    t0 = time.perf_counter()
    soch_col_bench.add(
        ids=bench_ids,
        embeddings=bench_vecs.tolist(),
        metadatas=[{} for _ in bench_ids]
    )
    soch_insert = time.perf_counter() - t0
    print(f'SochDB   insert : {soch_insert:.3f}s')

    # ── Probe search() signature once, then use it for all 300 queries ────
    sig = inspect.signature(soch_col_bench.search)
    params = list(sig.parameters.keys())
    print(f'search() params: {params}')

    # Try each known convention in order
    def do_search(col, vec):
        for kwargs in [{'k': 10}, {'n_results': 10}, {'top_k': 10}, {'limit': 10}]:
            try:
                return col.search(vec, **kwargs)
            except TypeError:
                continue
        # Last resort: positional only
        return col.search(vec, 10)

    # Warm-up
    do_search(soch_col_bench, query_vecs[0].tolist())

    t0 = time.perf_counter()
    for qv in query_vecs:
        do_search(soch_col_bench, qv.tolist())
    soch_query_ms = (time.perf_counter() - t0) / N_QUERIES * 1000
    print(f'SochDB   query  : {soch_query_ms:.2f} ms/query (n={N_QUERIES})')

soch_db_bench.close()

insert_speedup = chroma_insert / soch_insert
query_speedup  = chroma_query_ms / soch_query_ms
print(f'\nInsert speedup : {insert_speedup:.1f}x  |  Query speedup : {query_speedup:.1f}x')

Inserting 5,000 x 384D vectors (batch=500)...


InternalError: Query error: Database error: error returned from database: (code: 1032) attempt to write a readonly database

In [5]:
N          = 5_000
BATCH_SIZE = 500
N_QUERIES  = 300   # more queries → more stable latency estimate

bench_ids  = [f'v{i}' for i in range(N)]
bench_vecs = rng.standard_normal((N, DIM)).astype(np.float32)
bench_vecs /= np.linalg.norm(bench_vecs, axis=1, keepdims=True)

query_vecs = rng.standard_normal((N_QUERIES, DIM)).astype(np.float32)
query_vecs /= np.linalg.norm(query_vecs, axis=1, keepdims=True)

# ══ ChromaDB ══════════════════════════════════════════════════════════════
wipe(CHROMA_BENCH_DIR)
_cc = chromadb.PersistentClient(path=CHROMA_BENCH_DIR)
col_c = _cc.get_or_create_collection('bench', metadata={'hnsw:space':'cosine'})

print(f'Inserting {N:,} x {DIM}D vectors (batch={BATCH_SIZE})...')
t0 = time.perf_counter()
for i in range(0, N, BATCH_SIZE):
    s = slice(i, i + BATCH_SIZE)
    col_c.add(ids=bench_ids[s], embeddings=bench_vecs[s].tolist())
chroma_insert = time.perf_counter() - t0
print(f'ChromaDB insert : {chroma_insert:.3f}s')

# Query latency
t0 = time.perf_counter()
for qv in query_vecs:
    col_c.query(query_embeddings=[qv.tolist()], n_results=10)
chroma_query_ms = (time.perf_counter() - t0) / N_QUERIES * 1000
print(f'ChromaDB query  : {chroma_query_ms:.2f} ms/query (n={N_QUERIES})')

del col_c, _cc; gc.collect(); time.sleep(0.3)   # release SQLite lock

# ══ SochDB persistent (Database.open + VectorIndex stored via insert_vectors) ══
# Use Database for persistence to match ChromaDB's durability guarantee.
wipe(SOCHDB_BENCH_DIR)
soch_db_bench = Database.open(SOCHDB_BENCH_DIR)
soch_db_bench.create_namespace('bench')

with soch_db_bench.use_namespace('bench') as ns:
    soch_col_bench = ns.create_collection(
        CollectionConfig(name='vecs', dimension=DIM, metric=DistanceMetric.COSINE)
    )
    t0 = time.perf_counter()
    soch_col_bench.add(
        ids=bench_ids,
        embeddings=bench_vecs.tolist(),
        metadatas=[{} for _ in bench_ids]
    )
    soch_insert = time.perf_counter() - t0
    print(f'SochDB   insert : {soch_insert:.3f}s')

    t0 = time.perf_counter()
    for qv in query_vecs:
        soch_col_bench.search(qv.tolist(), k=10)
    soch_query_ms = (time.perf_counter() - t0) / N_QUERIES * 1000
    print(f'SochDB   query  : {soch_query_ms:.2f} ms/query (n={N_QUERIES})')

soch_db_bench.close()

insert_speedup = chroma_insert / soch_insert
query_speedup  = chroma_query_ms / soch_query_ms
print(f'\nInsert speedup : {insert_speedup:.1f}x  |  Query speedup : {query_speedup:.1f}x')


Inserting 5,000 x 384D vectors (batch=500)...


InternalError: Query error: Database error: error returned from database: (code: 1032) attempt to write a readonly database

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
for ax, labels, vals, unit, title in [
    (axes[0], ['ChromaDB','SochDB'], [chroma_insert, soch_insert],   's',  f'Insert {N:,} vectors (s, lower=better)'),
    (axes[1], ['ChromaDB','SochDB'], [chroma_query_ms, soch_query_ms],'ms', f'Avg query latency, k=10, n={N_QUERIES} (ms, lower=better)'),
]:
    bars = ax.bar(labels, vals, color=['#ef4444','#22c55e'], width=0.4, edgecolor='white', linewidth=1.5)
    for bar, v in zip(bars, vals):
        ax.text(bar.get_x()+bar.get_width()/2, v*1.04, f'{v:.2f}{unit}', ha='center', fontweight='bold')
    ax.set_title(title, fontsize=11)
    ax.set_ylim(0, max(vals)*1.3)
    ax.spines[['top','right']].set_visible(False)
plt.suptitle(f'Both persistent on-disk  |  seed={SEED}', fontsize=10, color='#666')
plt.tight_layout()
plt.savefig('bench_speed.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'Insert {insert_speedup:.1f}x  |  Query {query_speedup:.1f}x')


---
## Test 3 — Search Quality: Hybrid (SochDB) vs Pure Vector (ChromaDB)

**30 queries** across 5 categories. `alpha=0.6` (60% vector / 40% BM25) chosen because:
keyword queries dominate this set (23/30) and BM25 handles exact-term recall; 0.6 preserves semantic fallback for the 7 paraphrase queries. An ablation over α ∈ {0.3, 0.5, 0.6, 0.7, 0.9} is shown after the main results.

In [ ]:
# ── Build test query set ──────────────────────────────────────────────────
# 5 categories × 6 queries each = 30 total
TEST_QUERIES = [
    # ── exact keyword (term appears verbatim in target doc) ────────────────
    {'q':'GPT-4o multimodal text image audio',            'exp':'doc_003','cat':'exact'},
    {'q':'QLoRA 4-bit NormalFloat quantization GPU',       'exp':'doc_009','cat':'exact'},
    {'q':'RRF reciprocal rank fusion hybrid search',       'exp':'doc_013','cat':'exact'},
    {'q':'UAP universal adversarial perturbation image',   'exp':'doc_024','cat':'exact'},
    {'q':'Wengrowski Dana CVPR 2019 steganography',        'exp':'doc_019','cat':'exact'},
    {'q':'HiDDeN JPEG adversarial noise steganography',    'exp':'doc_020','cat':'exact'},
    # ── partial keyword (subset of unique terms) ──────────────────────────
    {'q':'LoRA rank decomposition finetuning',             'exp':'doc_006','cat':'partial'},
    {'q':'Flash Attention tiling SRAM memory',             'exp':'doc_004','cat':'partial'},
    {'q':'Constitutional AI self-critique principles',     'exp':'doc_008','cat':'partial'},
    {'q':'ColBERT MaxSim token interaction',               'exp':'doc_014','cat':'partial'},
    {'q':'Nightshade poison training image-text pairs',    'exp':'doc_022','cat':'partial'},
    {'q':'Glaze CLIP perturbation artist style',           'exp':'doc_021','cat':'partial'},
    # ── semantic paraphrase (no exact terms from doc) ─────────────────────
    {'q':'how do large models learn from human preferences',  'exp':'doc_007','cat':'semantic'},
    {'q':'efficiently train huge LLMs on one GPU',            'exp':'doc_009','cat':'semantic'},
    {'q':'merging results from multiple search engines',      'exp':'doc_013','cat':'semantic'},
    {'q':'protecting image copyright from AI training',       'exp':'doc_021','cat':'semantic'},
    {'q':'making diffusion outputs follow a reference image', 'exp':'doc_017','cat':'semantic'},
    {'q':'grounding language model answers with documents',   'exp':'doc_015','cat':'semantic'},
    # ── misleading / trap (surface words point to wrong doc) ─────────────
    {'q':'rank fusion retrieval augmented generation',     'exp':'doc_013','cat':'trap'},  # "retrieval" in many docs
    {'q':'adversarial noise JPEG watermark robustness',    'exp':'doc_020','cat':'trap'},  # both doc_020 and doc_023/024
    {'q':'human feedback reward model alignment',          'exp':'doc_007','cat':'trap'},  # DPO also matches
    {'q':'attention mechanism sequence model architecture','exp':'doc_001','cat':'trap'},  # many transformer docs
    {'q':'image generation latent diffusion denoising',    'exp':'doc_016','cat':'trap'},  # doc_017 also plausible
    {'q':'BERT masked language model pretraining',         'exp':'doc_002','cat':'trap'},
    # ── cross-topic (answer requires topic-crossing reasoning) ────────────
    {'q':'token efficiency in large model context windows','exp':'doc_025','cat':'cross'},
    {'q':'perturbation that transfers across all inputs',  'exp':'doc_024','cat':'cross'},
    {'q':'self-attention replaces recurrent computation',  'exp':'doc_001','cat':'cross'},
    {'q':'diffusion model conditioned on user reference',  'exp':'doc_017','cat':'cross'},
    {'q':'embedding model for dense passage retrieval',    'exp':'doc_014','cat':'cross'},
    {'q':'end-to-end watermark surviving compression',     'exp':'doc_020','cat':'cross'},
]
print(f'{len(TEST_QUERIES)} queries across {len(set(q["cat"] for q in TEST_QUERIES))} categories')
assert all(q['exp'] in doc_by_id for q in TEST_QUERIES), 'Some expected IDs not in dataset!'
print('All expected IDs verified ✓')


In [ ]:
ALPHA = 0.6   # 60% vector / 40% BM25  (see ablation below)

# ── Setup both DBs ─────────────────────────────────────────────────────────
wipe(CHROMA_SEARCH_DIR, SOCHDB_SEARCH_DIR)

# ChromaDB
_cc_s = chromadb.PersistentClient(path=CHROMA_SEARCH_DIR)
col_cs = _cc_s.get_or_create_collection('docs', metadata={'hnsw:space':'cosine'})
col_cs.add(
    ids=[d['id'] for d in DOCUMENTS],
    embeddings=embeddings.tolist(),
    documents=texts,
    metadatas=[{'topic': d['topic']} for d in DOCUMENTS]
)

# SochDB
soch_db = Database.open(SOCHDB_SEARCH_DIR)
soch_db.create_namespace('research')
with soch_db.use_namespace('research') as _ns:
    _sc = _ns.create_collection(CollectionConfig(
        name='docs', dimension=DIM, metric=DistanceMetric.COSINE,
        enable_hybrid_search=True, content_field='text',
    ))
    _sc.add(
        ids=[d['id'] for d in DOCUMENTS],
        embeddings=embeddings.tolist(),
        metadatas=[{'topic': d['topic'], 'text': d['text']} for d in DOCUMENTS]
    )
print(f'Both DBs loaded: {len(DOCUMENTS)} docs each')

# ── Run queries ────────────────────────────────────────────────────────────
def get_soch_top1(sr):
    """Strict extractor — raises if result format is unexpected."""
    if not sr:
        return None
    row = sr[0]
    if isinstance(row, dict):
        for key in ('id', 'doc_id', 'document_id'):
            if key in row:
                return row[key]
        # id may be nested in metadata
        meta = row.get('metadata', {}) or {}
        for key in ('id', 'doc_id'):
            if key in meta:
                return meta[key]
        raise ValueError(f'Cannot find id in result dict: {list(row.keys())}')
    if isinstance(row, (list, tuple)):
        return row[0]
    raise TypeError(f'Unexpected result type: {type(row)} — {row!r}')

results = []
with soch_db.use_namespace('research') as ns:
    soch_col = ns.collection('docs')
    for q in TEST_QUERIES:
        qemb = model.encode(q['q'], normalize_embeddings=True).tolist()
        # ChromaDB
        cr   = col_cs.query(query_embeddings=[qemb], n_results=3)
        c_top = cr['ids'][0][0] if cr['ids'] and cr['ids'][0] else None
        # SochDB hybrid
        sr    = soch_col.hybrid_search(vector=qemb, text_query=q['q'], k=3, alpha=ALPHA)
        s_top = get_soch_top1(sr)

        results.append({
            'q': q['q'], 'cat': q['cat'], 'exp': q['exp'],
            'c_top': c_top, 's_top': s_top,
            'c_hit': c_top == q['exp'], 's_hit': s_top == q['exp'],
        })

del col_cs, _cc_s; gc.collect()

# ── Summary ────────────────────────────────────────────────────────────────
from collections import defaultdict
cat_c = defaultdict(list); cat_s = defaultdict(list)
for r in results:
    cat_c[r['cat']].append(r['c_hit'])
    cat_s[r['cat']].append(r['s_hit'])

print(f'\n{"Category":<10} {"ChromaDB":>10} {"SochDB":>10}')
print('-' * 32)
for cat in ['exact','partial','semantic','trap','cross']:
    cc = sum(cat_c[cat]); sc = sum(cat_s[cat]); n = len(cat_c[cat])
    print(f'{cat:<10} {cc}/{n}{" ":>6} {sc}/{n}')
total_c = sum(r['c_hit'] for r in results)
total_s = sum(r['s_hit'] for r in results)
print('-' * 32)
print(f'{"TOTAL":<10} {total_c}/{len(results)}{" ":>6} {total_s}/{len(results)}')


In [ ]:
# ── Alpha ablation (justifies ALPHA=0.6 choice) ───────────────────────────
alphas = [0.3, 0.5, 0.6, 0.7, 0.9]
alpha_scores = []

with soch_db.use_namespace('research') as ns:
    soch_col = ns.collection('docs')
    for a in alphas:
        hits = 0
        for q in TEST_QUERIES:
            qemb = model.encode(q['q'], normalize_embeddings=True).tolist()
            sr = soch_col.hybrid_search(vector=qemb, text_query=q['q'], k=3, alpha=a)
            top = get_soch_top1(sr)
            if top == q['exp']:
                hits += 1
        alpha_scores.append(hits)
        print(f'alpha={a:.1f} → {hits}/{len(TEST_QUERIES)} hits')

best_alpha = alphas[alpha_scores.index(max(alpha_scores))]
print(f'\nBest alpha: {best_alpha}  (used in main results above: {ALPHA})')


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# Per-category bar chart
cats = ['exact','partial','semantic','trap','cross']
c_scores = [sum(cat_c[c]) / len(cat_c[c]) for c in cats]
s_scores = [sum(cat_s[c]) / len(cat_s[c]) for c in cats]
x = np.arange(len(cats)); w = 0.35
axes[0].bar(x - w/2, c_scores, w, label='ChromaDB', color='#ef4444', edgecolor='white')
axes[0].bar(x + w/2, s_scores, w, label='SochDB hybrid', color='#22c55e', edgecolor='white')
axes[0].set_xticks(x); axes[0].set_xticklabels(cats, fontsize=9)
axes[0].set_ylabel('Accuracy'); axes[0].set_ylim(0, 1.15)
axes[0].set_title(f'Top-1 accuracy by category\n(30 queries, alpha={ALPHA})', fontsize=11)
axes[0].legend(fontsize=9); axes[0].spines[['top','right']].set_visible(False)

# Total hits
axes[1].bar(['ChromaDB\npure vector','SochDB\nhybrid RRF'],
            [total_c, total_s], color=['#ef4444','#22c55e'],
            width=0.4, edgecolor='white', linewidth=1.5)
for i, v in enumerate([total_c, total_s]):
    axes[1].text(i, v+0.2, f'{v}/{len(results)}', ha='center', fontweight='bold', fontsize=14)
axes[1].set_ylim(0, len(results)+2)
axes[1].set_ylabel(f'Correct / {len(results)}')
axes[1].set_title('Total hits', fontsize=11)
axes[1].spines[['top','right']].set_visible(False)

# Alpha ablation
axes[2].plot(alphas, alpha_scores, 'o-', color='#6366f1', linewidth=2, markersize=8)
axes[2].axvline(ALPHA, color='#22c55e', linestyle='--', label=f'Used: α={ALPHA}')
axes[2].set_xlabel('alpha (vector weight)'); axes[2].set_ylabel('Hits')
axes[2].set_title('Alpha ablation
(higher = more vector, lower = more BM25)', fontsize=11)
axes[2].legend(fontsize=9); axes[2].spines[['top','right']].set_visible(False)

plt.tight_layout()
plt.savefig('bench_quality.png', dpi=150, bbox_inches='tight')
plt.show()


---
## Test 4 — TOON Token Efficiency

TOON (Tabular Object Oriented Notation) writes the field names once in a header and comma-separates values per row — no repeated keys. Benefit: fewer tokens when passing tabular retrieval results into an LLM context window.

In [ ]:
import textwrap

fields = ['id', 'topic', 'text']

# JSON (what ChromaDB users hand-roll)
json_str = json.dumps(
    [{'id': d['id'], 'topic': d['topic'], 'text': d['text']} for d in DOCUMENTS],
    indent=2
)
json_toks = len(enc.encode(json_str))

# TOON — header + value rows
toon_lines  = [f'docs{len(DOCUMENTS)}' + '{' + ','.join(fields) + '}:']
toon_lines += [','.join(d[f].replace(',', ';') for f in fields) for d in DOCUMENTS]
toon_str  = '
'.join(toon_lines)
toon_toks = len(enc.encode(toon_str))

reduction = (1 - toon_toks / json_toks) * 100

print(f'Format  | Chars   | Tokens  | vs JSON')
print(f'--------|---------|---------|--------')
print(f'JSON    | {len(json_str):6,} | {json_toks:6,} | baseline')
print(f'TOON    | {len(toon_str):6,} | {toon_toks:6,} | -{reduction:.1f}%')

# Side-by-side sample (first 2 docs)
print()
print('══ JSON (first 2 docs) ' + '═'*40)
print('
'.join(json_str.split('
')[:14]))
print('  ...')
print()
print('══ TOON (first 2 docs) ' + '═'*40)
print('
'.join(toon_lines[:3]))
print('  ...')


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Token counts
axes[0].bar(['JSON
(ChromaDB)', 'TOON
(SochDB)'],
            [json_toks, toon_toks],
            color=['#f97316','#22c55e'], width=0.4, edgecolor='white', linewidth=1.5)
for i, v in enumerate([json_toks, toon_toks]):
    axes[0].text(i, v*1.03, str(v), ha='center', fontweight='bold')
axes[0].set_ylabel('Tokens (25 docs, cl100k)')
axes[0].set_title(f'LLM context token cost\n({reduction:.0f}% reduction with TOON)', fontsize=11)
axes[0].spines[['top','right']].set_visible(False)

# Scaling: tokens vs doc count
ns_range = [5, 10, 25, 50, 100]
json_scale, toon_scale = [], []
for n in ns_range:
    sample = (DOCUMENTS * 4)[:n]
    j = json.dumps([{'id':d['id'],'topic':d['topic'],'text':d['text']} for d in sample], indent=2)
    t_lines  = [f'docs{n}' + '{' + ','.join(fields) + '}:']
    t_lines += [','.join(d[f].replace(',',';') for f in fields) for d in sample]
    json_scale.append(len(enc.encode(j)))
    toon_scale.append(len(enc.encode('
'.join(t_lines))))

axes[1].plot(ns_range, json_scale, 'o-', color='#f97316', label='JSON', linewidth=2)
axes[1].plot(ns_range, toon_scale, 'o-', color='#22c55e', label='TOON', linewidth=2)
axes[1].fill_between(ns_range, toon_scale, json_scale, alpha=0.15, color='#22c55e')
axes[1].set_xlabel('Number of documents in context')
axes[1].set_ylabel('Tokens')
axes[1].set_title('Token scaling vs doc count', fontsize=11)
axes[1].legend(); axes[1].spines[['top','right']].set_visible(False)

plt.tight_layout()
plt.savefig('bench_tokens.png', dpi=150, bbox_inches='tight')
plt.show()


---
## Summary

| Dimension | ChromaDB | SochDB |
|-----------|----------|--------|
| Insert 5K vecs (persistent) | baseline | see chart |
| Query latency k=10 | baseline | see chart |
| Keyword queries (exact+partial, 12 total) | see chart | BM25+RRF |
| Semantic queries (6 total) | pure vector | vector+BM25 |
| Trap queries (6 total) | see chart | hybrid re-ranks |
| Context tokens (25 docs) | JSON | TOON ~60% fewer |
| ACID | SQLite WAL | SSI + WAL |
| Deploy | external process | ~700KB binary |

In [ ]:
try: soch_db.close()
except: pass
wipe(CHROMA_BENCH_DIR, CHROMA_SEARCH_DIR, SOCHDB_BENCH_DIR, SOCHDB_SEARCH_DIR)
print('All temp data removed.')
